# LUMEN-299 · 1 — One constant, and what it is a measurement of

**The failure.** In 1941 Raymond Birge pooled the best measurements of the speed of light
made since 1926 and reported **299,776 ± 4 km/s**. The value the metre has been defined
by since 1983 is 299,792.458. His interval missed by four of its own standard errors, and
every number that went into it was carefully reduced and honestly reported.

Nothing about that failure is arithmetic. It is about what each number was a number *of*
— light in air or in vacuo, a league of which length, an instrument sharing its
systematic error with the instrument next door — and none of that is recoverable from a
column of values and a column of uncertainties. This notebook does the part that precedes
any pooling: it says what the quantity is, registers what each laboratory meant by it,
and files every experiment under whose it is.

| | reaches into |
|---|---|
| the unit that decides the third figure | `core.UNITS`, `core.BASES` |
| the quantity, as a typed estimand | `estimands.Estimand`, `Estimand.transfer_to` |
| whose measurement it is | `io.Catalog`, `io.Program` |
| what each laboratory meant by *c* | `io.DefinitionRegistry`, `io.Consensus` |

In [ ]:
import sys

sys.path.insert(0, ".")

import math

import numpy as np
import pandas as pd

import lightspeed as L

from axiom.core import UNITS
from axiom.estimands import Estimand, TransferPlan
from axiom.io import Catalog, DefinitionRegistry, Program

from axiom.display import enable, show, table

sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import caption, intervals, mark_x

enable();  # every axiom result renders itself from here on

print(f"c, as defined in 1983       {L.C_TRUE:>14,.3f} km/s")
print(f"c in dry air at 0 C, 1 atm  {L.C_AIR_TRUE:>14,.3f} km/s  (refractive index {L.N_AIR})")
print(f"the difference               {L.C_TRUE - L.C_AIR_TRUE:>13,.1f} km/s")

## 1 · Two numbers for one experiment

Fizeau, 1849: a toothed wheel, a mirror on Montmartre 8.6 km away, and a published
result of **70,948 leagues per second**. Textbooks give it as 313,300 km/s and as
315,300 km/s, and both are right — they took different leagues. `axiom.core` has no idea
what a league is until one is declared, which is the point: a unit is a fact to be
registered, not a footnote to be remembered.

In [ ]:
km_per_second, line = UNITS.convert(L.FIZEAU_LEAGUES, "league", "km")
show(line)
other_league = 4.416  # the league some later accounts assumed, in km
table(
    [["lieue de 25 au degré", f"{float(L.LEAGUE_KM):.4f}", f"{km_per_second:,.0f}"],
     ["the 4.416 km league", f"{other_league:.4f}", f"{L.FIZEAU_LEAGUES * other_league:,.0f}"],
     ["the truth, in air", "", f"{L.C_AIR_TRUE:,.0f}"]],
    headers=("league", "km", "km/s"),
    title="the same 70,948 leagues per second",
)
print("units the system knows about:", UNITS.units())

The difference between the two leagues is 2,000 km/s — larger than Fizeau's whole error
and larger than Foucault's disagreement with him thirteen years later. The conversion is
one line and it leaves a `LedgerLine`; everything downstream of this notebook that quotes
Fizeau in km/s is quoting that line.

## 2 · What the number is a number of

An `Estimand` in `axiom` is a counterfactual quantity: what happens to an outcome when a
treatment is set. A constant of nature is not the response to anything, and no
`QuantityKind` describes a *level* without pretence. So the honest move is to declare what
every instrument in this programme actually does. It sets a **transit time** and reads a
**distance**, and the ratio of the two is the speed of light:

$$c = \frac{\text{distance covered}}{\text{one second of transit}}$$

That is a `ratio` estimand with the treatment `transit` and the outcome `distance`, and
the two things the century's records differ in — the medium the light crossed, and
whether it was light at all rather than a radio wave — are exactly the facets the type
already has: the intervention's `version` and the `population`.

In [ ]:
c_vacuo: Estimand = L.estimand()
c_air = L.estimand(medium="air")
c_radio = L.estimand(radiation="radio")
print("estimand   :", c_vacuo.name, "|", c_vacuo.quantity.kind, "| dimension", c_vacuo.dimension)
print("treatment  :", c_vacuo.treatment.name, f"({c_vacuo.treatment.unit})",
      "| outcome:", c_vacuo.outcome.name, f"({c_vacuo.outcome.unit})")
print("hash       :", c_vacuo.content_hash()[:16])
print("in air differs in:", c_vacuo.differing_facets(c_air),
      "| a radio measurement differs in:", c_vacuo.differing_facets(c_radio))
c_vacuo

In [ ]:
plan: TransferPlan = c_air.transfer_to(c_vacuo)
print("reading an in-air number as in-vacuo:", plan.status)
table(
    [[e.facet, "blocked" if e.blocked else "assumed",
      "; ".join(a.name for a in e.assumptions) or "—"] for e in plan.entries],
    headers=("facet", "status", "under"),
)
print("\nversion_irrelevance is the assumption that the medium does not matter.")
print(f"It matters by {L.C_TRUE - L.C_AIR_TRUE:.1f} km/s, which is larger than every standard")
print("error reported after 1926. The type cannot know that; the registry below can.")

The transfer is `downgraded`, not `blocked`: the facet table knows the medium is a
*version* of the same intervention and names the assumption that reading across it makes.
It cannot know the assumption is false by 88 km/s. That knowledge lives one level up, in
what each laboratory *meant* — section 4.

## 3 · The century, as published

Sixteen experiments, four kinds of instrument. The `sigma_off` column is how many of its
own reported standard errors each published value sits from the truth *in its own
medium*, which is the first thing to know about a corpus and the last thing its authors
could have.

In [ ]:
ledger = L.ledger_frame()
table(
    [[int(r.year), r.who, r.family, r.medium, f"{r.reported:,.1f}", f"{r.reported_se:,.4g}",
      f"{r.sigma_off:+.1f}"] for r in ledger.itertuples()],
    headers=("year", "who", "family", "medium", "reported", "± (reported)", "sigma off"),
    title="the published record, against the truth in each experiment's own medium",
)
late = ledger[ledger.year >= 1900]
print(f"experiments after 1900 more than two of their own standard errors from the truth: "
      f"{int((late.sigma_off.abs() > 2).sum())} of {len(late)}")

In [ ]:
rows = []
for e in L.HISTORY:
    if e.year < 1900:
        continue
    truth = L.C_AIR_TRUE if e.medium == "air" else L.C_TRUE
    rows.append((f"{e.year} {e.who}", e.reported - truth,
                 e.reported - truth - 2 * e.reported_se, e.reported - truth + 2 * e.reported_se))
fig = intervals(
    rows, ref=0.0, ref_label="the truth",
    title="Seventy years of published values, each ±2 of its own standard error",
    subtitle="distance from the truth in the experiment's own medium, km/s — the 1928–41 cluster sits low together",
    x_title="published value − truth (km/s)",
)
caption(fig, "Every interval is honest about the scatter its authors could see. What none "
             "of them can show is the thing they share: between 1928 and 1941 five "
             "instruments of two kinds agree with each other and not with the value "
             "the metre was later defined by.")

## 4 · Whose measurement it is

A programme is a scope. `Program(party, program)` here reads *party* as the instrument
family — the thing whose systematic errors travel together — and *program* as the
laboratory. Every artifact the case study files goes into exactly one of these, and
leaving one is a ledgered transfer (notebook 5).

In [ ]:
import tempfile
from pathlib import Path

catalog = Catalog(Path(tempfile.mkdtemp()) / "lumen")
programs = L.programs()
for program in programs.values():
    catalog.register(program)

table([[p.scope, L.FAMILY_LABEL[p.party]] for p in catalog.programs()],
      headers=("scope", "instrument family"), title="registered programmes")

# Each laboratory files the estimand it actually measured, in its own scope.
for e in L.HISTORY:
    store = catalog.store(e.program)
    store.put(L.estimand(medium=e.medium, radiation=e.radiation),
              label=f"{e.key}:estimand", tags={"experiment": e.key, "medium": e.medium})
found = catalog.find(type_name="Estimand")
print(f"\n{len(found)} estimands filed across {len(catalog.scopes())} scopes;",
      f"{sum(1 for f in found if f.tags['medium'] == 'air')} of them measured in air")

## 5 · What each laboratory meant by *c*

The scope says whose number it is. It does not say what the number is *of*, and for this
constant that question has two live answers — in air, or corrected to vacuo — that differ
by more than any modern error bar. A `DefinitionRegistry` records, per scope, the spec a
laboratory has agreed the term means, and `consensus` asks whether several laboratories
currently mean the same thing.

Three answers are possible, and the third is the one that matters: a laboratory that has
**never registered** the term has not agreed with anybody.

In [ ]:
definitions = DefinitionRegistry(catalog)
labs = {e.lab: e for e in L.HISTORY}
for lab, e in labs.items():
    if lab == "perrotin":
        continue  # Nice never wrote down which it meant
    if lab == "michelson":
        definitions.register(e.program, "c", L.estimand(medium="air"), at="1879-06-01",
                             note="the Annapolis reductions, before the air correction")
        definitions.register(e.program, "c", L.estimand(medium="vacuo"), at="1879-09-01",
                             note="corrected to vacuo for publication")
        continue
    definitions.register(e.program, "c", L.estimand(medium=e.medium), at=f"{e.year}-01-01",
                         note=f"as published: in {e.medium}")

silent = definitions.consensus("c", list(programs.values()))
print(silent.summary())
print(" verdict:", silent.verdict().status, "—", silent.verdict().reason[:100])

In [ ]:
definitions.register(labs["perrotin"].program, "c", L.estimand(medium="air"), at="1902-01-01")
split = definitions.consensus("c", list(programs.values()))
print("verdict once everybody has spoken:", split.verdict().status)
in_air = L.estimand(medium="air").content_hash()
table(
    [[digest[:12] + "…", ", ".join(sorted(scopes)), "in air" if digest == in_air else "in vacuo"]
     for digest, scopes in split.by_digest.items()],
    headers=("definition", "scopes that mean it", "which is"),
    title="two definitions of c, and who holds each",
)

Michelson is the instructive case. His 1879 paper reported 299,910 km/s in air and
299,944 corrected to vacuo, and the correction was the last thing he did. A registry that
versions by *content* records that as a change with a date and the field that moved,
rather than as two numbers with the same name.

In [ ]:
michelson = programs["michelson"]
table(
    [[d.version, d.registered, d.note or "—"] for d in definitions.history(michelson, "c")],
    headers=("version", "registered", "note"), title="rotating-mirror/michelson: c",
)
for change in definitions.changes(michelson, "c"):
    print(f"{change.at}: v{change.from_version} -> v{change.to_version}")
    for field_, moved in sorted(change.changed.items()):
        print(f"   {field_}: {moved}")

## 6 · The world these notebooks run in

Every number after this point is computed in a synthetic world calibrated to the record
above: the truth is 299,792.458 km/s, each *generation* of instrument carries one
systematic error drawn once, each experiment adds its own and reports the standard error
its authors reported. The two generations active between 1928 and 1945 draw their
systematics around the same centre, because that is what the record says happened and
because no hierarchy over families can see it — which notebook 4 is about.

In [ ]:
world = L.simulate()
frame = world.frame()
table(
    [[int(r.year), r.who, r.generation, f"{r.read_vacuo:,.1f}", f"{r.se:,.4g}",
      f"{(r.read_vacuo - L.C_TRUE) / r.se:+.1f}", f"{r.published:,.0f}"]
     for r in frame.itertuples()],
    headers=("year", "who", "generation", "read (vacuo)", "±", "sigma off", "published"),
    title="the century, as this world ran it",
)
table([[g, f"{v:+.2f}"] for g, v in world.generation_offsets.items()],
      headers=("generation", "systematic (km/s)"), title="what each generation carried, unknown to it")

## What notebook 1 established

1. **The third significant figure of the first measurement is a unit**, and a unit is a
   registered fact with a conversion and a ledger line, not something a reader is trusted
   to remember. Fizeau's 70,948 leagues are 315,300 km/s or 313,300 depending on the
   league, and `UNITS.convert` says which was used.
2. **A constant is not a counterfactual**, so the estimand is declared for what every
   instrument does — distance covered per second of transit, a `ratio` — with the medium
   as the intervention's version and the radiation as the population. Reading an in-air
   number as in-vacuo is a `downgraded` transfer under `version_irrelevance`, and the
   assumption is false by 88 km/s.
3. **Every experiment has a scope**, party = instrument family, programme = laboratory.
4. **Silence is not agreement.** With one laboratory unregistered the consensus on *c* is
   `unverified`; once it speaks, the century holds two definitions and the verdict is
   `blocked`. Michelson's own change from air to vacuo is a versioned diff with a date.

Notebook 2 designs the instrument that produced the best of these numbers.